# Campaign 008 T0 — detectability gate

No training. Base-model probe: can the runtime representation separate H2O from cardinality-matched random and scrambled keep sets?
Commit `eca272951e896ae7864f52b18374eaa34bf05cfc`, r=0.5, 160 dev prompts.

In [ ]:
import json, os, platform, shutil, subprocess, sys, time
from pathlib import Path

REPO = "https://github.com/kartikbindra/btp-backdoor-research"
COMMIT = "eca272951e896ae7864f52b18374eaa34bf05cfc"
WORK = Path("/kaggle/working")
SRC = Path("/kaggle/tmp/btp-research")
gpu = subprocess.check_output(["nvidia-smi","--query-gpu=name,uuid,driver_version,compute_cap","--format=csv,noheader"], text=True).strip()
print(gpu); print(sys.version); print(platform.platform())
assert "Tesla" in gpu or "NVIDIA" in gpu
SRC.parent.mkdir(parents=True, exist_ok=True)
if SRC.exists():
    shutil.rmtree(SRC)
subprocess.run(["git","clone","--quiet",REPO,str(SRC)], check=True)
subprocess.run(["git","-C",str(SRC),"checkout","--quiet",COMMIT], check=True)
head = subprocess.check_output(["git","-C",str(SRC),"rev-parse","HEAD"], text=True).strip()
print("repo head", head)
for rel in ("src/pfseb/runtime_train.py","src/pfseb/campaign008.py","scripts/run_runtime_train.py","scripts/run_runtime_probe.py"):
    p = SRC / rel
    assert p.exists(), f"missing {rel} at {head}"
import torch, transformers
print("torch", torch.__version__, "transformers", transformers.__version__, "cuda", torch.cuda.is_available())
(WORK/"session_env.json").write_text(json.dumps({"gpu": gpu, "torch": torch.__version__, "transformers": transformers.__version__, "repo_head": head}, indent=2))


In [ ]:
def run_phase(tag, args, out_dir=None, max_minutes=0):
    target = out_dir or str(WORK / "rt_c008_t0_probe")
    cmd = [sys.executable, "-u", "-m"] + args
    if max_minutes:
        cmd += ["--max_minutes", str(max_minutes)]
    print("RUN", " ".join(cmd), flush=True)
    started = time.time()
    with (WORK / f"{tag}.log").open("w", buffering=1) as log:
        proc = subprocess.run(cmd, cwd=str(SRC), stdout=log, stderr=subprocess.STDOUT)
    print(tag, "exit", proc.returncode, "minutes", round((time.time()-started)/60, 1), flush=True)
    return proc.returncode


In [ ]:
OUT = WORK / "rt_c008_t0_probe"; OUT.mkdir(parents=True, exist_ok=True)
rc = run_phase("probe", ["scripts.run_runtime_probe","--out_dir",str(OUT),"--max_items","160",
    "--retention",str(0.5),"--bins","long_512"], max_minutes=120)
print("probe exit", rc)
if (OUT/"summary.json").exists():
    print((OUT/"summary.json").read_text()[:3000])

In [ ]:
stamp = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
bundle = WORK / f"rt_c008_t0_probe_{stamp}"
bundle.mkdir(exist_ok=True)
for root in [p for p in WORK.iterdir() if p.is_dir() and p.name != bundle.name]:
    if root.name in {"smoke", "rt_c008_t0_probe"}:
        shutil.copytree(root, bundle / root.name, dirs_exist_ok=True)
for item in WORK.iterdir():
    if item.is_file() and (item.suffix in {".json",".log"} or item.name.endswith(".jsonl")):
        shutil.copy2(item, bundle / item.name)
shutil.make_archive(str(WORK / "rt_c008_t0_probe_outputs"), "zip", bundle)
print("packaged", bundle, sorted(p.name for p in bundle.iterdir()))
